<a href="https://colab.research.google.com/github/eshan14git/football-qa-nlp/blob/disath-dev/notebooks/football_qa_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Football Natural Language Question Answering Application

## Notebook Overview

This notebook implements the final integrated application for the Football Natural Language
Question Answering project.

Two trained intent-classification models are available for comparison:

1. Word-and-character TF-IDF Logistic Regression
2. Word-level one-dimensional Convolutional Neural Network

Logistic Regression is selected as the final model because it achieved 74% accuracy on the
independent manual robustness set, compared with 66% for the CNN.

The integrated application:

1. classifies the user's question intent;
2. extracts teams, player names, and dates;
3. searches the finalized Version 2 football records; and
4. returns a factual answer.

When a question matches multiple historical games, the application requests a more specific
date instead of guessing.

## Connect Drive and import libraries

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import re
import os
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

from tensorflow.keras.preprocessing.sequence import pad_sequences

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

print("TensorFlow version:", tf.__version__)
print("Demo runtime ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow version: 2.20.0
Demo runtime ready.


## Configure Model Artifact Location

The interface requires the saved Logistic Regression and CNN artifacts. Their directory is
configured through `CUSTOM_MODEL_DIRECTORY`.

When the artifacts are stored in the default Google Drive location, this value can remain
`None`. A different user can set it to the folder containing their copies of the model files
without changing any other code.

Required artifacts:

- `disa_word_character_tfidf_vectorizer.joblib`
- `disa_logistic_regression_intent_model.joblib`
- `disa_cnn_intent_model.keras`
- `disa_cnn_tokenizer.pkl`
- `disa_cnn_label_encoder.joblib`
- `disa_cnn_config.json`

## Define and validate artifact paths

In [2]:
# ------------------------------------------------------------------
# MODEL ARTIFACT CONFIGURATION
# ------------------------------------------------------------------
# Change only this value if the model files are stored elsewhere.
#
# Examples:
# "/content/drive/MyDrive/NLP/models"
# "/content/drive/MyDrive/football-qa-models"
# "/content/football-qa-nlp/models"
#
# Leave as None to use the default Google Drive location.

CUSTOM_MODEL_DIRECTORY = None

DEFAULT_MODEL_DIRECTORY = (
    "/content/drive/MyDrive/NLP/models"
)

MODEL_DIRECTORY = (
    CUSTOM_MODEL_DIRECTORY
    if CUSTOM_MODEL_DIRECTORY
    else DEFAULT_MODEL_DIRECTORY
)

MODEL_DIRECTORY = os.path.abspath(
    os.path.expanduser(MODEL_DIRECTORY)
)

print("Selected model directory:")
print(MODEL_DIRECTORY)


# ------------------------------------------------------------------
# ARTIFACT FILENAMES
# ------------------------------------------------------------------

ARTIFACT_FILENAMES = {
    "ML vectorizer": (
        "disa_word_character_tfidf_vectorizer.joblib"
    ),
    "ML model": (
        "disa_logistic_regression_intent_model.joblib"
    ),
    "CNN model": (
        "disa_cnn_intent_model.keras"
    ),
    "CNN tokenizer": (
        "disa_cnn_tokenizer.pkl"
    ),
    "CNN label encoder": (
        "disa_cnn_label_encoder.joblib"
    ),
    "CNN configuration": (
        "disa_cnn_config.json"
    )
}


# ------------------------------------------------------------------
# BUILD COMPLETE PATHS
# ------------------------------------------------------------------

artifact_paths = {
    artifact_name: os.path.join(
        MODEL_DIRECTORY,
        artifact_filename
    )
    for artifact_name, artifact_filename
    in ARTIFACT_FILENAMES.items()
}

ML_VECTORIZER_PATH = artifact_paths[
    "ML vectorizer"
]

ML_MODEL_PATH = artifact_paths[
    "ML model"
]

CNN_MODEL_PATH = artifact_paths[
    "CNN model"
]

CNN_TOKENIZER_PATH = artifact_paths[
    "CNN tokenizer"
]

CNN_LABEL_ENCODER_PATH = artifact_paths[
    "CNN label encoder"
]

CNN_CONFIG_PATH = artifact_paths[
    "CNN configuration"
]


# ------------------------------------------------------------------
# VALIDATE FILES
# ------------------------------------------------------------------

print("\nMODEL ARTIFACT VALIDATION")
print("-" * 70)

missing_artifacts = []

for artifact_name, artifact_path in artifact_paths.items():
    artifact_exists = os.path.isfile(
        artifact_path
    )

    status = (
        "FOUND"
        if artifact_exists
        else "MISSING"
    )

    print(
        f"{artifact_name:<22} "
        f"{status:<8} "
        f"{artifact_path}"
    )

    if not artifact_exists:
        missing_artifacts.append(
            {
                "name": artifact_name,
                "path": artifact_path
            }
        )

if missing_artifacts:
    missing_file_list = "\n".join(
        f"- {artifact['name']}: {artifact['path']}"
        for artifact in missing_artifacts
    )

    raise FileNotFoundError(
        "\nOne or more required model artifacts "
        "could not be found.\n\n"
        "Change CUSTOM_MODEL_DIRECTORY near the "
        "top of this cell.\n\n"
        f"Missing artifacts:\n{missing_file_list}"
    )

print("\nAll required model artifacts were found.")

Selected model directory:
/content/drive/MyDrive/NLP/models

MODEL ARTIFACT VALIDATION
----------------------------------------------------------------------
ML vectorizer          FOUND    /content/drive/MyDrive/NLP/models/disa_word_character_tfidf_vectorizer.joblib
ML model               FOUND    /content/drive/MyDrive/NLP/models/disa_logistic_regression_intent_model.joblib
CNN model              FOUND    /content/drive/MyDrive/NLP/models/disa_cnn_intent_model.keras
CNN tokenizer          FOUND    /content/drive/MyDrive/NLP/models/disa_cnn_tokenizer.pkl
CNN label encoder      FOUND    /content/drive/MyDrive/NLP/models/disa_cnn_label_encoder.joblib
CNN configuration      FOUND    /content/drive/MyDrive/NLP/models/disa_cnn_config.json

All required model artifacts were found.


## Load the ML and CNN Models

Each trained model is loaded together with its required preprocessing components. Their intent
classes and CNN configuration are inspected to confirm compatibility before prediction
functions are created.

## Load all artifacts

In [3]:
ml_vectorizer = joblib.load(
    ML_VECTORIZER_PATH
)

ml_model = joblib.load(
    ML_MODEL_PATH
)

cnn_model = tf.keras.models.load_model(
    CNN_MODEL_PATH
)

with open(
    CNN_TOKENIZER_PATH,
    "rb"
) as tokenizer_file:
    cnn_tokenizer = pickle.load(
        tokenizer_file
    )

cnn_label_encoder = joblib.load(
    CNN_LABEL_ENCODER_PATH
)

with open(
    CNN_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as configuration_file:
    cnn_configuration = json.load(
        configuration_file
    )

print("MODEL ARTIFACTS LOADED")
print("-" * 60)
print(
    "ML vectorizer type:",
    type(ml_vectorizer).__name__
)
print(
    "ML model type:",
    type(ml_model).__name__
)
print(
    "CNN model name:",
    cnn_model.name
)
print(
    "CNN input shape:",
    cnn_model.input_shape
)
print(
    "CNN output shape:",
    cnn_model.output_shape
)

MODEL ARTIFACTS LOADED
------------------------------------------------------------
ML vectorizer type: FeatureUnion
ML model type: LogisticRegression
CNN model name: football_intent_cnn
CNN input shape: (None, 25)
CNN output shape: (None, 5)


## Verify intent classes

In [4]:
ml_intent_classes = list(
    ml_model.classes_
)

cnn_intent_classes = list(
    cnn_label_encoder.classes_
)

configured_cnn_classes = (
    cnn_configuration["intent_classes"]
)

print("INTENT-CLASS COMPATIBILITY")
print("-" * 60)

print("\nML classes:")
for class_index, intent in enumerate(
    ml_intent_classes
):
    print(f"{class_index}: {intent}")

print("\nCNN classes:")
for class_index, intent in enumerate(
    cnn_intent_classes
):
    print(f"{class_index}: {intent}")

print(
    "\nConfigured maximum sequence length:",
    cnn_configuration["max_sequence_length"]
)

assert ml_intent_classes == cnn_intent_classes
assert cnn_intent_classes == configured_cnn_classes

assert cnn_model.output_shape[-1] == len(
    cnn_intent_classes
)

print(
    "\nBoth models use the same intent classes "
    "in the same order."
)

INTENT-CLASS COMPATIBILITY
------------------------------------------------------------

ML classes:
0: match_score
1: match_scorers
2: match_winner
3: player_match_goal_count
4: player_match_scoring_minutes

CNN classes:
0: match_score
1: match_scorers
2: match_winner
3: player_match_goal_count
4: player_match_scoring_minutes

Configured maximum sequence length: 25

Both models use the same intent classes in the same order.


## Configure the Football QA Dataset

The integrated application uses the finalized Version 2 master dataset as its factual knowledge
source. The selected intent-classification model identifies the type of question, while the
retrieval layer uses match, team, player, and date metadata to locate the appropriate answer.

The dataset path is configurable so that another user can run the application using a different
Google Drive or local project location.

## Configure and load dataset

In [5]:
# ------------------------------------------------------------------
# QA DATASET CONFIGURATION
# ------------------------------------------------------------------
# Change this value when the Version 2 master dataset is stored
# somewhere other than the default Google Drive location.
#
# Leave as None to use the default path.

CUSTOM_QA_DATASET_PATH = None

DEFAULT_QA_DATASET_PATH = (
    "/content/drive/MyDrive/NLP/"
    "exports_v2/football_qa_v2_master.csv"
)

QA_DATASET_PATH = (
    CUSTOM_QA_DATASET_PATH
    if CUSTOM_QA_DATASET_PATH
    else DEFAULT_QA_DATASET_PATH
)

QA_DATASET_PATH = os.path.abspath(
    os.path.expanduser(QA_DATASET_PATH)
)

print("Selected QA dataset:")
print(QA_DATASET_PATH)

if not os.path.isfile(QA_DATASET_PATH):
    raise FileNotFoundError(
        "\nThe Version 2 master dataset was not found.\n"
        "Change CUSTOM_QA_DATASET_PATH to the correct location.\n"
        f"Current path: {QA_DATASET_PATH}"
    )

qa_master_df = pd.read_csv(
    QA_DATASET_PATH,
    low_memory=False
)

print("\nFOOTBALL QA MASTER DATASET LOADED")
print("-" * 70)
print(
    "Rows:",
    f"{len(qa_master_df):,}"
)
print(
    "Columns:",
    len(qa_master_df.columns)
)
print(
    "Unique matches:",
    f"{qa_master_df['match_id'].nunique():,}"
)
print(
    "Intent classes:",
    qa_master_df["intent"].nunique()
)

print("\nIntent distribution:")

display(
    qa_master_df["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="question_count")
)

print("\nAvailable columns:")
print(qa_master_df.columns.tolist())

Selected QA dataset:
/content/drive/MyDrive/NLP/exports_v2/football_qa_v2_master.csv

FOOTBALL QA MASTER DATASET LOADED
----------------------------------------------------------------------
Rows: 194,159
Columns: 25
Unique matches: 49,481
Intent classes: 5

Intent distribution:


,intent,question_count
0,match_winner,49481
1,match_score,49481
2,player_match_goal_count,39931
3,player_match_scoring_minutes,39758
4,match_scorers,15508



Available columns:
['question_id', 'semantic_question_id', 'match_id', 'intent', 'question', 'answer', 'template_id', 'template_family', 'template_seen_in_training', 'evaluation_track', 'question_variant', 'date_format', 'team_order', 'source_dataset', 'scorer', 'team', 'player_goal_count', 'goal_count', 'scoring_minutes', 'date_standardized', 'home_team_standardized', 'away_team_standardized', 'home_score', 'away_score', 'tournament']


## Validate retrieval fields

In [6]:
REQUIRED_RETRIEVAL_COLUMNS = {
    "match_id",
    "intent",
    "question",
    "answer",
    "date_standardized",
    "home_team_standardized",
    "away_team_standardized",
    "home_score",
    "away_score"
}

missing_retrieval_columns = (
    REQUIRED_RETRIEVAL_COLUMNS.difference(
        qa_master_df.columns
    )
)

if missing_retrieval_columns:
    raise ValueError(
        "The QA master dataset is missing required columns: "
        f"{sorted(missing_retrieval_columns)}"
    )

qa_master_df["date_standardized"] = pd.to_datetime(
    qa_master_df["date_standardized"],
    errors="coerce"
)

print("RETRIEVAL DATA VALIDATION")
print("-" * 70)
print(
    "Missing standardized dates:",
    qa_master_df["date_standardized"].isna().sum()
)
print(
    "Missing home teams:",
    qa_master_df[
        "home_team_standardized"
    ].isna().sum()
)
print(
    "Missing away teams:",
    qa_master_df[
        "away_team_standardized"
    ].isna().sum()
)
print(
    "Missing answers:",
    qa_master_df["answer"].isna().sum()
)

valid_dates = qa_master_df[
    "date_standardized"
].dropna()

print(
    "\nDate range:",
    valid_dates.min().date(),
    "to",
    valid_dates.max().date()
)

assert not missing_retrieval_columns
assert qa_master_df[
    "date_standardized"
].notna().all()

print("\nThe dataset is ready for factual retrieval.")

RETRIEVAL DATA VALIDATION
----------------------------------------------------------------------
Missing standardized dates: 0
Missing home teams: 0
Missing away teams: 0
Missing answers: 0

Date range: 1872-11-30 to 2026-07-19

The dataset is ready for factual retrieval.


## Build Factual Retrieval Tables

The master dataset contains multiple question types derived from the same football records.
Compact lookup tables are created for match facts, match scorers, player goal counts, and
player scoring minutes.

These tables retain one factual record per relevant match or player-match combination and
remove unnecessary generated-question variants.

## Build lookup tables

In [7]:
# One record for every match
match_lookup_df = (
    qa_master_df.loc[
        qa_master_df["intent"] == "match_winner",
        [
            "match_id",
            "date_standardized",
            "home_team_standardized",
            "away_team_standardized",
            "home_score",
            "away_score",
            "tournament"
        ]
    ]
    .drop_duplicates(
        subset=["match_id"]
    )
    .reset_index(drop=True)
)

# One aggregated scorer answer for matches with scorer data
match_scorers_lookup_df = (
    qa_master_df.loc[
        qa_master_df["intent"] == "match_scorers",
        [
            "match_id",
            "date_standardized",
            "home_team_standardized",
            "away_team_standardized",
            "answer"
        ]
    ]
    .drop_duplicates(
        subset=["match_id"]
    )
    .reset_index(drop=True)
)

# One goal-count record for each player and match
player_goal_count_lookup_df = (
    qa_master_df.loc[
        qa_master_df["intent"]
        == "player_match_goal_count",
        [
            "match_id",
            "date_standardized",
            "home_team_standardized",
            "away_team_standardized",
            "scorer",
            "team",
            "player_goal_count",
            "answer"
        ]
    ]
    .drop_duplicates(
        subset=[
            "match_id",
            "scorer"
        ]
    )
    .reset_index(drop=True)
)

# One scoring-minute record for each player and match
player_scoring_minutes_lookup_df = (
    qa_master_df.loc[
        qa_master_df["intent"]
        == "player_match_scoring_minutes",
        [
            "match_id",
            "date_standardized",
            "home_team_standardized",
            "away_team_standardized",
            "scorer",
            "team",
            "scoring_minutes",
            "answer"
        ]
    ]
    .drop_duplicates(
        subset=[
            "match_id",
            "scorer"
        ]
    )
    .reset_index(drop=True)
)

print("FACTUAL LOOKUP TABLES CREATED")
print("-" * 70)
print(
    "Match records:",
    f"{len(match_lookup_df):,}"
)
print(
    "Match-scorer records:",
    f"{len(match_scorers_lookup_df):,}"
)
print(
    "Player goal-count records:",
    f"{len(player_goal_count_lookup_df):,}"
)
print(
    "Player scoring-minute records:",
    f"{len(player_scoring_minutes_lookup_df):,}"
)

FACTUAL LOOKUP TABLES CREATED
----------------------------------------------------------------------
Match records: 49,481
Match-scorer records: 15,508
Player goal-count records: 39,931
Player scoring-minute records: 39,758


## Validate lookup tables

In [8]:
print("LOOKUP TABLE VALIDATION")
print("-" * 70)

print(
    "Duplicate match IDs in match table:",
    match_lookup_df["match_id"].duplicated().sum()
)

print(
    "Duplicate match IDs in scorer table:",
    match_scorers_lookup_df[
        "match_id"
    ].duplicated().sum()
)

print(
    "Duplicate player-match goal-count records:",
    player_goal_count_lookup_df.duplicated(
        subset=[
            "match_id",
            "scorer"
        ]
    ).sum()
)

print(
    "Duplicate player-match minute records:",
    player_scoring_minutes_lookup_df.duplicated(
        subset=[
            "match_id",
            "scorer"
        ]
    ).sum()
)

print("\nMissing scorer names:")
print(
    "Match goal counts:",
    player_goal_count_lookup_df[
        "scorer"
    ].isna().sum()
)
print(
    "Scoring minutes:",
    player_scoring_minutes_lookup_df[
        "scorer"
    ].isna().sum()
)

assert len(match_lookup_df) == (
    qa_master_df["match_id"].nunique()
)

assert not match_lookup_df[
    "match_id"
].duplicated().any()

assert not match_scorers_lookup_df[
    "match_id"
].duplicated().any()

assert not player_goal_count_lookup_df.duplicated(
    subset=[
        "match_id",
        "scorer"
    ]
).any()

assert not player_scoring_minutes_lookup_df.duplicated(
    subset=[
        "match_id",
        "scorer"
    ]
).any()

print("\nAll factual lookup tables passed validation.")

LOOKUP TABLE VALIDATION
----------------------------------------------------------------------
Duplicate match IDs in match table: 0
Duplicate match IDs in scorer table: 0
Duplicate player-match goal-count records: 0
Duplicate player-match minute records: 0

Missing scorer names:
Match goal counts: 0
Scoring minutes: 0

All factual lookup tables passed validation.


## Inspect representative records

In [9]:
print("SAMPLE MATCH RECORDS")
display(
    match_lookup_df.head(3)
)

print("\nSAMPLE MATCH-SCORER RECORDS")
display(
    match_scorers_lookup_df.head(3)
)

print("\nSAMPLE PLAYER GOAL-COUNT RECORDS")
display(
    player_goal_count_lookup_df.head(3)
)

print("\nSAMPLE PLAYER SCORING-MINUTE RECORDS")
display(
    player_scoring_minutes_lookup_df.head(3)
)

SAMPLE MATCH RECORDS


,match_id,date_standardized,home_team_standardized,away_team_standardized,home_score,away_score,tournament
0,MATCH_00001,1872-11-30,Scotland,England,0,0,Friendly
1,MATCH_00003,1874-03-07,Scotland,England,2,1,Friendly
2,MATCH_00004,1875-03-06,England,Scotland,2,2,Friendly



SAMPLE MATCH-SCORER RECORDS


,match_id,date_standardized,home_team_standardized,away_team_standardized,answer
0,MATCH_00480,1916-07-02,Chile,Uruguay,José Piendibene scored 2 goals for Uruguay and...
1,MATCH_00482,1916-07-06,Argentina,Chile,"Alberto Ohaco scored 2 goals for Argentina, Te..."
2,MATCH_00484,1916-07-10,Argentina,Brazil,José Durand Laguna scored 1 goal for Argentina...



SAMPLE PLAYER GOAL-COUNT RECORDS


,match_id,date_standardized,home_team_standardized,away_team_standardized,scorer,team,player_goal_count,answer
0,MATCH_00480,1916-07-02,Chile,Uruguay,José Piendibene,Uruguay,2.0,José Piendibene scored 2 goals for Uruguay in ...
1,MATCH_00480,1916-07-02,Chile,Uruguay,Isabelino Gradín,Uruguay,2.0,Isabelino Gradín scored 2 goals for Uruguay in...
2,MATCH_00482,1916-07-06,Argentina,Chile,Alberto Ohaco,Argentina,2.0,Alberto Ohaco scored 2 goals for Argentina in ...



SAMPLE PLAYER SCORING-MINUTE RECORDS


,match_id,date_standardized,home_team_standardized,away_team_standardized,scorer,team,scoring_minutes,answer
0,MATCH_00480,1916-07-02,Chile,Uruguay,Isabelino Gradín,Uruguay,55' and 70',Isabelino Gradín scored at 55' and 70'.
1,MATCH_00480,1916-07-02,Chile,Uruguay,José Piendibene,Uruguay,44' and 75',José Piendibene scored at 44' and 75'.
2,MATCH_00482,1916-07-06,Argentina,Chile,Alberto Marcovecchio,Argentina,67' and 81',Alberto Marcovecchio scored at 67' and 81'.


## Extract Football Entities from Natural-Language Questions

Factual retrieval requires more than intent classification. The system must identify the teams,
player, and date mentioned in a question.

Team and player vocabularies are derived only from the finalized football dataset. Text is
normalized for case, accents, punctuation, and repeated whitespace before entity matching.
Longer names are prioritized to reduce partial-name collisions.

## Build team and player vocabularies

In [10]:
import unicodedata
from dateutil import parser as date_parser


def normalize_entity_text(text):
    """
    Normalize text for team and player entity matching.
    """
    text = str(text).casefold()

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        character
        for character in text
        if not unicodedata.combining(character)
    )

    text = text.replace("’", "'")

    text = re.sub(
        r"[^a-z0-9\s'-]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


team_names = sorted(
    set(
        match_lookup_df[
            "home_team_standardized"
        ].dropna().astype(str)
    ).union(
        set(
            match_lookup_df[
                "away_team_standardized"
            ].dropna().astype(str)
        )
    )
)

player_names = sorted(
    set(
        player_goal_count_lookup_df[
            "scorer"
        ].dropna().astype(str)
    ).union(
        set(
            player_scoring_minutes_lookup_df[
                "scorer"
            ].dropna().astype(str)
        )
    )
)

normalized_team_to_original = {
    normalize_entity_text(team): team
    for team in team_names
}

normalized_player_to_original = {
    normalize_entity_text(player): player
    for player in player_names
}

normalized_team_names = sorted(
    normalized_team_to_original.keys(),
    key=len,
    reverse=True
)

normalized_player_names = sorted(
    normalized_player_to_original.keys(),
    key=len,
    reverse=True
)

print("ENTITY VOCABULARIES CREATED")
print("-" * 70)
print(
    "Unique teams:",
    f"{len(team_names):,}"
)
print(
    "Unique goal scorers:",
    f"{len(player_names):,}"
)

print("\nSample teams:")
print(team_names[:20])

print("\nSample players:")
print(player_names[:20])

ENTITY VOCABULARIES CREATED
----------------------------------------------------------------------
Unique teams: 336
Unique goal scorers: 14,863

Sample teams:
['Abkhazia', 'Afghanistan', 'Albania', 'Alderney', 'Algeria', 'Ambazonia', 'American Samoa', 'Andalusia', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Arameans Suryoye', 'Argentina', 'Armenia', 'Artsakh', 'Aruba', 'Asturias', 'Australia', 'Austria']

Sample players:
["A'ala Hubail", 'A. Elangovan', 'Aage Rou Jensen', 'Aamir Abdallah', 'Aaran Lines', 'Aaron Boupendza', 'Aaron Hughes', 'Aaron Long', 'Aaron Mooy', 'Aaron Njovu', 'Aaron Ramsey', 'Aaron Tumwa', 'Aarón Padilla Gutiérrez', 'Aarón Suárez', 'Aatef Jenyat', 'Aavo Sillandi', 'Abat Aymbetov', 'Abay Bokoleyev', 'Abbas Ahmed Atwi', 'Abbas Chahrour']


## Extract non-overlapping names

In [11]:
def extract_non_overlapping_entities(
    question,
    normalized_names,
    normalized_to_original,
    maximum_entities=None
):
    """
    Find known names in normalized question text while preventing
    shorter names from overlapping longer matched names.
    """
    normalized_question = normalize_entity_text(
        question
    )

    candidate_matches = []

    for normalized_name in normalized_names:
        pattern = (
            r"(?<![a-z0-9])"
            + re.escape(normalized_name)
            + r"(?![a-z0-9])"
        )

        for match in re.finditer(
            pattern,
            normalized_question
        ):
            candidate_matches.append(
                {
                    "start": match.start(),
                    "end": match.end(),
                    "length": (
                        match.end()
                        - match.start()
                    ),
                    "normalized_name": normalized_name,
                    "original_name": (
                        normalized_to_original[
                            normalized_name
                        ]
                    )
                }
            )

    candidate_matches = sorted(
        candidate_matches,
        key=lambda item: (
            -item["length"],
            item["start"]
        )
    )

    accepted_matches = []

    for candidate in candidate_matches:
        overlaps_existing = any(
            candidate["start"] < accepted["end"]
            and candidate["end"] > accepted["start"]
            for accepted in accepted_matches
        )

        duplicate_entity = any(
            candidate["original_name"]
            == accepted["original_name"]
            for accepted in accepted_matches
        )

        if not overlaps_existing and not duplicate_entity:
            accepted_matches.append(candidate)

    accepted_matches = sorted(
        accepted_matches,
        key=lambda item: item["start"]
    )

    extracted_entities = [
        match["original_name"]
        for match in accepted_matches
    ]

    if maximum_entities is not None:
        extracted_entities = extracted_entities[
            :maximum_entities
        ]

    return extracted_entities


def extract_teams(question):
    """
    Extract up to two known teams from a question.
    """
    return extract_non_overlapping_entities(
        question=question,
        normalized_names=normalized_team_names,
        normalized_to_original=(
            normalized_team_to_original
        ),
        maximum_entities=2
    )


def extract_player(question):
    """
    Extract the first known scorer name from a question.
    """
    extracted_players = (
        extract_non_overlapping_entities(
            question=question,
            normalized_names=(
                normalized_player_names
            ),
            normalized_to_original=(
                normalized_player_to_original
            ),
            maximum_entities=1
        )
    )

    return (
        extracted_players[0]
        if extracted_players
        else None
    )

## Extract dates and years

In [12]:
MONTH_PATTERN = (
    r"(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|"
    r"apr(?:il)?|may|jun(?:e)?|jul(?:y)?|"
    r"aug(?:ust)?|sep(?:tember)?|oct(?:ober)?|"
    r"nov(?:ember)?|dec(?:ember)?)"
)

FULL_DATE_PATTERNS = [
    # 2014-07-08
    r"\b\d{4}-\d{1,2}-\d{1,2}\b",

    # 08/07/2014 or 08-07-2014
    r"\b\d{1,2}[/-]\d{1,2}[/-]\d{4}\b",

    # 8 July 2014
    rf"\b\d{{1,2}}\s+{MONTH_PATTERN}\s+\d{{4}}\b",

    # July 8, 2014
    rf"\b{MONTH_PATTERN}\s+\d{{1,2}}(?:st|nd|rd|th)?"
    rf",?\s+\d{{4}}\b"
]


def extract_question_date(question):
    """
    Extract a complete date from a question.

    ISO dates are parsed explicitly as year-month-day.
    Numeric slash dates are interpreted as day-month-year.
    Written month dates are parsed using dateutil.
    """
    question_lower = str(question).lower()

    # ISO format: 2014-07-08
    iso_date_match = re.search(
        r"\b\d{4}-\d{1,2}-\d{1,2}\b",
        question_lower
    )

    if iso_date_match:
        try:
            return pd.Timestamp(
                pd.to_datetime(
                    iso_date_match.group(0),
                    format="%Y-%m-%d"
                ).date()
            )

        except (ValueError, OverflowError):
            pass

    # Numeric day-first format: 08/07/2014
    numeric_date_match = re.search(
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{4}\b",
        question_lower
    )

    if numeric_date_match:
        try:
            parsed_date = date_parser.parse(
                numeric_date_match.group(0),
                dayfirst=True,
                fuzzy=False
            )

            return pd.Timestamp(
                parsed_date.date()
            )

        except (ValueError, OverflowError):
            pass

    # Written format: 8 July 2014
    day_month_year_match = re.search(
        rf"\b\d{{1,2}}\s+{MONTH_PATTERN}\s+\d{{4}}\b",
        question_lower,
        flags=re.IGNORECASE
    )

    if day_month_year_match:
        try:
            parsed_date = date_parser.parse(
                day_month_year_match.group(0),
                dayfirst=True,
                fuzzy=False
            )

            return pd.Timestamp(
                parsed_date.date()
            )

        except (ValueError, OverflowError):
            pass

    # Written format: July 8, 2014
    month_day_year_match = re.search(
        (
            rf"\b{MONTH_PATTERN}\s+"
            rf"\d{{1,2}}(?:st|nd|rd|th)?,?\s+"
            rf"\d{{4}}\b"
        ),
        question_lower,
        flags=re.IGNORECASE
    )

    if month_day_year_match:
        try:
            cleaned_date_text = re.sub(
                r"(\d)(st|nd|rd|th)\b",
                r"\1",
                month_day_year_match.group(0)
            )

            parsed_date = date_parser.parse(
                cleaned_date_text,
                dayfirst=False,
                fuzzy=False
            )

            return pd.Timestamp(
                parsed_date.date()
            )

        except (ValueError, OverflowError):
            pass

    return None


def extract_question_year(question):
    """
    Extract a four-digit year when present.
    """
    year_match = re.search(
        r"\b(18|19|20)\d{2}\b",
        str(question)
    )

    return (
        int(year_match.group(0))
        if year_match
        else None
    )

## Test entity extraction

In [13]:
entity_test_questions = [
    (
        "Who won between Brazil and Germany "
        "on 8 July 2014?"
    ),
    (
        "What was the score when France played "
        "Argentina on 2022-12-18?"
    ),
    (
        "How many goals did Lionel Messi score "
        "when Argentina played France on "
        "18 December 2022?"
    ),
    (
        "At what minutes did Kylian Mbappé score "
        "for France against Argentina in 2022?"
    )
]

entity_test_results = []

for question in entity_test_questions:
    entity_test_results.append(
        {
            "question": question,
            "teams": extract_teams(question),
            "player": extract_player(question),
            "date": extract_question_date(question),
            "year": extract_question_year(question)
        }
    )

entity_test_results_df = pd.DataFrame(
    entity_test_results
)

print("ENTITY EXTRACTION VERIFICATION")
print("-" * 70)

display(entity_test_results_df)

ENTITY EXTRACTION VERIFICATION
----------------------------------------------------------------------


,question,teams,player,date,year
0,Who won between Brazil and Germany on 8 July 2...,"[Brazil, Germany]",None,2014-07-08,2014
1,What was the score when France played Argentin...,"[France, Argentina]",None,2022-12-18,2022
2,How many goals did Lionel Messi score when Arg...,"[Argentina, France]",Lionel Messi,2022-12-18,2022
3,At what minutes did Kylian Mbappé score for Fr...,"[France, Argentina]",Kylian Mbappé,NaT,2022


## Retrieve Factual Answers

The retrieval system filters football records using the extracted teams, player, and date.
When one record matches, its factual answer is returned. When several historical matches remain,
the system requests a more specific date instead of guessing.

Answers stored in the finalized master dataset are reused to ensure consistency with the
validated QA-generation pipeline.

## Add winner and score answers to the match table

In [14]:
winner_answer_lookup_df = (
    qa_master_df.loc[
        qa_master_df["intent"] == "match_winner",
        [
            "match_id",
            "answer"
        ]
    ]
    .drop_duplicates(
        subset=["match_id"]
    )
    .rename(
        columns={
            "answer": "winner_answer"
        }
    )
)

score_answer_lookup_df = (
    qa_master_df.loc[
        qa_master_df["intent"] == "match_score",
        [
            "match_id",
            "answer"
        ]
    ]
    .drop_duplicates(
        subset=["match_id"]
    )
    .rename(
        columns={
            "answer": "score_answer"
        }
    )
)

match_lookup_df = (
    match_lookup_df
    .merge(
        winner_answer_lookup_df,
        on="match_id",
        how="left"
    )
    .merge(
        score_answer_lookup_df,
        on="match_id",
        how="left"
    )
)

print("MATCH ANSWERS ATTACHED")
print("-" * 70)
print(
    "Missing winner answers:",
    match_lookup_df[
        "winner_answer"
    ].isna().sum()
)
print(
    "Missing score answers:",
    match_lookup_df[
        "score_answer"
    ].isna().sum()
)

display(
    match_lookup_df[
        [
            "match_id",
            "date_standardized",
            "home_team_standardized",
            "away_team_standardized",
            "winner_answer",
            "score_answer"
        ]
    ].head(3)
)

assert match_lookup_df[
    "winner_answer"
].notna().all()

assert match_lookup_df[
    "score_answer"
].notna().all()

MATCH ANSWERS ATTACHED
----------------------------------------------------------------------
Missing winner answers: 0
Missing score answers: 0


,match_id,date_standardized,home_team_standardized,away_team_standardized,winner_answer,score_answer
0,MATCH_00001,1872-11-30,Scotland,England,The match between Scotland and England ended i...,England 0-0 Scotland.
1,MATCH_00003,1874-03-07,Scotland,England,Scotland won against England 2-1 on 1874-03-07.,England 1-2 Scotland.
2,MATCH_00004,1875-03-06,England,Scotland,The match between England and Scotland ended i...,Scotland 2-2 England.


## Match-context filtering

In [15]:
def filter_by_match_context(
    dataframe,
    teams,
    question_date,
    question_year
):
    """
    Filter records using extracted teams and date information.
    """
    candidates_df = dataframe.copy()

    for team in teams:
        candidates_df = candidates_df.loc[
            (
                candidates_df[
                    "home_team_standardized"
                ] == team
            )
            |
            (
                candidates_df[
                    "away_team_standardized"
                ] == team
            )
        ]

    if question_date is not None:
        candidates_df = candidates_df.loc[
            candidates_df[
                "date_standardized"
            ] == question_date
        ]

    elif question_year is not None:
        candidates_df = candidates_df.loc[
            candidates_df[
                "date_standardized"
            ].dt.year == question_year
        ]

    return candidates_df.reset_index(
        drop=True
    )


def format_candidate_matches(candidates_df):
    """
    Produce a concise list of possible historical matches.
    """
    unique_matches_df = (
        candidates_df[
            [
                "match_id",
                "date_standardized",
                "home_team_standardized",
                "away_team_standardized"
            ]
        ]
        .drop_duplicates(
            subset=["match_id"]
        )
        .sort_values(
            "date_standardized"
        )
    )

    candidate_descriptions = []

    for _, match_row in unique_matches_df.head(10).iterrows():
        candidate_descriptions.append(
            (
                f"{match_row['date_standardized'].date()}: "
                f"{match_row['home_team_standardized']} vs "
                f"{match_row['away_team_standardized']}"
            )
        )

    return candidate_descriptions

## Main factual-retrieval function

In [16]:
def retrieve_football_answer(
    question,
    predicted_intent
):
    """
    Retrieve a factual football answer using the predicted intent
    and entities extracted from the user's question.
    """
    teams = extract_teams(question)
    player = extract_player(question)
    question_date = extract_question_date(
        question
    )
    question_year = extract_question_year(
        question
    )

    extracted_entities = {
        "teams": teams,
        "player": player,
        "date": (
            str(question_date.date())
            if question_date is not None
            else None
        ),
        "year": question_year
    }

    match_level_intents = {
        "match_winner",
        "match_score",
        "match_scorers"
    }

    player_level_intents = {
        "player_match_goal_count",
        "player_match_scoring_minutes"
    }

    if predicted_intent in match_level_intents:
        if len(teams) < 2:
            return {
                "status": "missing_information",
                "answer": (
                    "Please include both football teams "
                    "in the question."
                ),
                "entities": extracted_entities,
                "candidate_matches": []
            }

        match_candidates_df = (
            filter_by_match_context(
                dataframe=match_lookup_df,
                teams=teams,
                question_date=question_date,
                question_year=question_year
            )
        )

        unique_match_candidates_df = (
            match_candidates_df
            .drop_duplicates(
                subset=["match_id"]
            )
        )

        if len(unique_match_candidates_df) == 0:
            return {
                "status": "not_found",
                "answer": (
                    "No matching football record was found. "
                    "Check the team names and date."
                ),
                "entities": extracted_entities,
                "candidate_matches": []
            }

        if len(unique_match_candidates_df) > 1:
            candidate_matches = (
                format_candidate_matches(
                    unique_match_candidates_df
                )
            )

            return {
                "status": "ambiguous",
                "answer": (
                    f"I found "
                    f"{len(unique_match_candidates_df)} "
                    "matches involving those teams. "
                    "Please include the exact match date."
                ),
                "entities": extracted_entities,
                "candidate_matches": candidate_matches
            }

        selected_match = (
            unique_match_candidates_df.iloc[0]
        )

        if predicted_intent == "match_winner":
            answer = selected_match[
                "winner_answer"
            ]

        elif predicted_intent == "match_score":
            answer = selected_match[
                "score_answer"
            ]

        else:
            scorer_candidates_df = (
                match_scorers_lookup_df.loc[
                    match_scorers_lookup_df[
                        "match_id"
                    ] == selected_match["match_id"]
                ]
            )

            if len(scorer_candidates_df) == 1:
                answer = scorer_candidates_df.iloc[0][
                    "answer"
                ]

            elif (
                selected_match["home_score"]
                + selected_match["away_score"]
                == 0
            ):
                answer = (
                    f"No goals were scored when "
                    f"{selected_match['home_team_standardized']} "
                    f"played "
                    f"{selected_match['away_team_standardized']} "
                    f"on "
                    f"{selected_match['date_standardized'].date()}."
                )

            else:
                answer = (
                    "The match was found, but scorer "
                    "information is unavailable."
                )

        return {
            "status": "answered",
            "answer": answer,
            "entities": extracted_entities,
            "candidate_matches": []
        }

    if predicted_intent in player_level_intents:
        if player is None:
            return {
                "status": "missing_information",
                "answer": (
                    "Please include a recognized player name "
                    "in the question."
                ),
                "entities": extracted_entities,
                "candidate_matches": []
            }

        if predicted_intent == "player_match_goal_count":
            player_lookup_df = (
                player_goal_count_lookup_df
            )
        else:
            player_lookup_df = (
                player_scoring_minutes_lookup_df
            )

        player_candidates_df = (
            player_lookup_df.loc[
                player_lookup_df["scorer"]
                == player
            ]
            .copy()
        )

        player_candidates_df = (
            filter_by_match_context(
                dataframe=player_candidates_df,
                teams=teams,
                question_date=question_date,
                question_year=question_year
            )
        )

        if len(player_candidates_df) == 0:
            return {
                "status": "not_found",
                "answer": (
                    "No matching player scoring record was "
                    "found. Include the teams and match date "
                    "to make the question more specific."
                ),
                "entities": extracted_entities,
                "candidate_matches": []
            }

        unique_player_matches_df = (
            player_candidates_df
            .drop_duplicates(
                subset=[
                    "match_id",
                    "scorer"
                ]
            )
        )

        if len(unique_player_matches_df) > 1:
            candidate_matches = (
                format_candidate_matches(
                    unique_player_matches_df
                )
            )

            return {
                "status": "ambiguous",
                "answer": (
                    f"I found "
                    f"{len(unique_player_matches_df)} "
                    f"matching scoring records for {player}. "
                    "Please include the exact teams or date."
                ),
                "entities": extracted_entities,
                "candidate_matches": candidate_matches
            }

        answer = unique_player_matches_df.iloc[0][
            "answer"
        ]

        return {
            "status": "answered",
            "answer": answer,
            "entities": extracted_entities,
            "candidate_matches": []
        }

    return {
        "status": "unsupported_intent",
        "answer": (
            "The predicted question type is not supported "
            "by the retrieval system."
        ),
        "entities": extracted_entities,
        "candidate_matches": []
    }

## Test factual retrieval directly

In [17]:
retrieval_test_cases = [
    {
        "question": (
            "Who won between Brazil and Germany "
            "on 8 July 2014?"
        ),
        "intent": "match_winner"
    },
    {
        "question": (
            "What was the score when Brazil played "
            "Germany on 2014-07-08?"
        ),
        "intent": "match_score"
    },
    {
        "question": (
            "Who scored when Brazil faced Germany "
            "on 8 July 2014?"
        ),
        "intent": "match_scorers"
    },
    {
        "question": (
            "How many goals did Lionel Messi score "
            "when Argentina played France on "
            "18 December 2022?"
        ),
        "intent": "player_match_goal_count"
    },
    {
        "question": (
            "At what minutes did Kylian Mbappé score "
            "for France against Argentina on "
            "18 December 2022?"
        ),
        "intent": (
            "player_match_scoring_minutes"
        )
    },
    {
        "question": (
            "Who won between Brazil and Germany?"
        ),
        "intent": "match_winner"
    }
]

retrieval_test_results = []

for test_case in retrieval_test_cases:
    retrieval_result = retrieve_football_answer(
        question=test_case["question"],
        predicted_intent=test_case["intent"]
    )

    retrieval_test_results.append(
        {
            "question": test_case["question"],
            "intent": test_case["intent"],
            "status": retrieval_result["status"],
            "answer": retrieval_result["answer"],
            "candidate_matches": (
                retrieval_result[
                    "candidate_matches"
                ]
            )
        }
    )

retrieval_test_results_df = pd.DataFrame(
    retrieval_test_results
)

print("FACTUAL RETRIEVAL VERIFICATION")
print("-" * 70)

display(retrieval_test_results_df)

FACTUAL RETRIEVAL VERIFICATION
----------------------------------------------------------------------


,question,intent,status,answer,candidate_matches
0,Who won between Brazil and Germany on 8 July 2...,match_winner,answered,Germany won against Brazil 7-1 on 2014-07-08.,[]
1,What was the score when Brazil played Germany ...,match_score,answered,Germany 7-1 Brazil.,[]
2,Who scored when Brazil faced Germany on 8 July...,match_scorers,answered,"Thomas Müller scored 1 goal for Germany, Miros...",[]
3,How many goals did Lionel Messi score when Arg...,player_match_goal_count,answered,Lionel Messi scored 2 goals for Argentina in t...,[]
4,At what minutes did Kylian Mbappé score for Fr...,player_match_scoring_minutes,answered,"Kylian Mbappé scored at 80', 81', and 118'.",[]
5,Who won between Brazil and Germany?,match_winner,ambiguous,I found 23 matches involving those teams. Plea...,"[1963-05-05: Germany vs Brazil, 1965-06-06: Br..."


## Define Model-Specific Text Preparation

The Logistic Regression model and CNN use different preprocessing procedures. The ML model
uses lowercase text with restricted punctuation, matching its training notebook. The CNN uses
minimal whitespace normalization before tokenization.

Each new question must be transformed using the preprocessing method associated with the
selected model.

## Define both preprocessing function

In [18]:
def clean_ml_question_text(text):
    """
    Apply the same text cleaning used by the ML notebook.
    """
    text = str(text).lower()

    # Normalize apostrophes
    text = text.replace("’", "'")

    # Remove punctuation while retaining letters,
    # numbers, apostrophes and hyphens
    text = re.sub(
        r"[^a-z0-9\s'-]",
        " ",
        text
    )

    # Replace repeated whitespace with one space
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def prepare_cnn_text(text):
    """
    Apply the same minimal normalization used by the CNN notebook.
    """
    if pd.isna(text):
        return ""

    normalized_text = str(text).strip()

    normalized_text = " ".join(
        normalized_text.split()
    )

    return normalized_text

## Verify preprocessing

In [19]:
preprocessing_example = (
    "Wht was the final scor in Brazil versus Chile?"
)

ml_prepared_example = clean_ml_question_text(
    preprocessing_example
)

cnn_prepared_example = prepare_cnn_text(
    preprocessing_example
)

print("PREPROCESSING VERIFICATION")
print("-" * 60)
print("Original:")
print(preprocessing_example)

print("\nML-prepared:")
print(ml_prepared_example)

print("\nCNN-prepared:")
print(cnn_prepared_example)

assert ml_prepared_example == (
    "wht was the final scor in brazil versus chile"
)

assert cnn_prepared_example == preprocessing_example

print("\nBoth preprocessing functions work correctly.")

PREPROCESSING VERIFICATION
------------------------------------------------------------
Original:
Wht was the final scor in Brazil versus Chile?

ML-prepared:
wht was the final scor in brazil versus chile

CNN-prepared:
Wht was the final scor in Brazil versus Chile?

Both preprocessing functions work correctly.


## Define Prediction Functions

Separate prediction functions apply the correct preprocessing and model-specific
transformations. Each function returns the predicted intent and the model's maximum class
probability as its confidence score.

## Logistic Regression prediction function

In [20]:
def predict_ml_intent(question):
    """
    Predict one question's intent using Logistic Regression.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError(
            "The question must contain non-empty text."
        )

    cleaned_question = clean_ml_question_text(
        question
    )

    question_features = ml_vectorizer.transform(
        [cleaned_question]
    )

    probability_vector = ml_model.predict_proba(
        question_features
    )[0]

    predicted_class_index = int(
        np.argmax(probability_vector)
    )

    predicted_intent = ml_model.classes_[
        predicted_class_index
    ]

    confidence = float(
        probability_vector[
            predicted_class_index
        ]
    )

    return {
        "model": "Logistic Regression",
        "predicted_intent": predicted_intent,
        "confidence": confidence
    }

## CNN prediction function

In [21]:
def predict_cnn_intent(question):
    """
    Predict one question's intent using the CNN.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError(
            "The question must contain non-empty text."
        )

    prepared_question = prepare_cnn_text(
        question
    )

    question_sequence = (
        cnn_tokenizer.texts_to_sequences(
            [prepared_question]
        )
    )

    padded_question = pad_sequences(
        question_sequence,
        maxlen=cnn_configuration[
            "max_sequence_length"
        ],
        padding="post",
        truncating="post"
    )

    probability_vector = cnn_model.predict(
        padded_question,
        verbose=0
    )[0]

    predicted_class_index = int(
        np.argmax(probability_vector)
    )

    predicted_intent = (
        cnn_label_encoder.inverse_transform(
            [predicted_class_index]
        )[0]
    )

    confidence = float(
        probability_vector[
            predicted_class_index
        ]
    )

    return {
        "model": "1D CNN",
        "predicted_intent": predicted_intent,
        "confidence": confidence
    }

## Test both models

In [22]:
test_questions = [
    "Who won between Brazil and Germany?",
    "What was the final score in France versus Argentina?",
    "Who scored in the Spain and Italy match?",
    "How many goals did Lionel Messi score?",
    "At what minutes did Cristiano Ronaldo score?"
]

comparison_results = []

for question in test_questions:
    ml_result = predict_ml_intent(
        question
    )

    cnn_result = predict_cnn_intent(
        question
    )

    comparison_results.append(
        {
            "question": question,
            "ML prediction": (
                ml_result["predicted_intent"]
            ),
            "ML confidence": (
                ml_result["confidence"]
            ),
            "CNN prediction": (
                cnn_result["predicted_intent"]
            ),
            "CNN confidence": (
                cnn_result["confidence"]
            ),
            "models_agree": (
                ml_result["predicted_intent"]
                == cnn_result["predicted_intent"]
            )
        }
    )

comparison_results_df = pd.DataFrame(
    comparison_results
)

print("MODEL PREDICTION VERIFICATION")
print("-" * 70)

display(
    comparison_results_df.style.format({
        "ML confidence": "{:.4f}",
        "CNN confidence": "{:.4f}"
    })
)

MODEL PREDICTION VERIFICATION
----------------------------------------------------------------------


,question,ML prediction,ML confidence,CNN prediction,CNN confidence,models_agree
0,Who won between Brazil and Germany?,match_winner,0.4670,match_winner,0.9815,True
1,What was the final score in France versus Argentina?,match_score,0.9319,match_score,1.0000,True
2,Who scored in the Spain and Italy match?,match_scorers,0.8253,match_scorers,1.0000,True
3,How many goals did Lionel Messi score?,player_match_goal_count,0.9025,player_match_goal_count,1.0000,True
4,At what minutes did Cristiano Ronaldo score?,player_match_scoring_minutes,0.8915,player_match_scoring_minutes,1.0000,True


## Interactive Football Question Answering Interface

The Gradio interface accepts a natural-language football question and returns a factual answer
from the finalized dataset.

Logistic Regression is used as the selected final intent model for answer retrieval. Users can
also display the CNN prediction or compare both models' intent predictions and confidence
scores.

The interface displays extracted entities and reports when a question is ambiguous, missing
required information, or cannot be matched to the available football records.

## Install and import Gradio

In [23]:
!pip install -q gradio

import gradio as gr

print("Gradio version:", gr.__version__)

Gradio version: 6.20.0


## Interface prediction handler

In [24]:
def run_integrated_football_qa(
    question,
    model_choice
):
    """
    Display selected model predictions and retrieve a factual
    answer using Logistic Regression as the selected final model.
    """
    if not isinstance(question, str) or not question.strip():
        empty_predictions_df = pd.DataFrame(
            [
                {
                    "Model": "No model executed",
                    "Predicted Intent": (
                        "Enter a football question"
                    ),
                    "Confidence": 0.0
                }
            ]
        )

        return (
            empty_predictions_df,
            "Please enter a football question.",
            "No entities were extracted."
        )

    # Logistic Regression is always executed because it is the
    # selected final model for factual-answer retrieval.
    ml_result = predict_ml_intent(
        question
    )

    prediction_results = []

    if model_choice in [
        "Logistic Regression",
        "Compare Both"
    ]:
        prediction_results.append(
            {
                "Model": ml_result["model"],
                "Predicted Intent": (
                    ml_result["predicted_intent"]
                ),
                "Confidence": round(
                    ml_result["confidence"],
                    4
                )
            }
        )

    if model_choice in [
        "1D CNN",
        "Compare Both"
    ]:
        cnn_result = predict_cnn_intent(
            question
        )

        prediction_results.append(
            {
                "Model": cnn_result["model"],
                "Predicted Intent": (
                    cnn_result["predicted_intent"]
                ),
                "Confidence": round(
                    cnn_result["confidence"],
                    4
                )
            }
        )

    predictions_df = pd.DataFrame(
        prediction_results
    )

    retrieval_result = retrieve_football_answer(
        question=question,
        predicted_intent=(
            ml_result["predicted_intent"]
        )
    )

    factual_answer = retrieval_result[
        "answer"
    ]

    entities = retrieval_result[
        "entities"
    ]

    teams_text = (
        ", ".join(entities["teams"])
        if entities["teams"]
        else "Not detected"
    )

    player_text = (
        entities["player"]
        if entities["player"]
        else "Not detected"
    )

    date_text = (
        entities["date"]
        if entities["date"]
        else (
            str(entities["year"])
            if entities["year"]
            else "Not detected"
        )
    )

    candidate_matches = retrieval_result[
        "candidate_matches"
    ]

    if candidate_matches:
        candidate_text = "\n".join(
            f"- {candidate_match}"
            for candidate_match
            in candidate_matches
        )
    else:
        candidate_text = "None"

    retrieval_details = f"""
### Retrieval Details

- **Selected final model:** Logistic Regression
- **Intent used for retrieval:** `{ml_result["predicted_intent"]}`
- **Retrieval status:** `{retrieval_result["status"]}`
- **Teams detected:** {teams_text}
- **Player detected:** {player_text}
- **Date/year detected:** {date_text}

### Possible Matches

{candidate_text}
"""

    return (
        predictions_df,
        factual_answer,
        retrieval_details
    )

## Build and launch the interface

In [25]:
benchmark_results_df = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "1D CNN"
        ],
        "Generated-Test Accuracy": [
            "100%",
            "100%"
        ],
        "Generated-Test Macro F1": [
            "1.0000",
            "1.0000"
        ],
        "Manual Accuracy": [
            "74%",
            "66%"
        ],
        "Manual Macro F1": [
            "0.7381",
            "0.6571"
        ]
    }
)


with gr.Blocks(
    title="Football Natural Language Question Answering"
) as demo:

    gr.Markdown(
        """
        # Football Natural Language Question Answering

        Ask a factual question about an international football match.

        The system supports:

        - Match winner
        - Match score
        - Match goal scorers
        - A player's goal count
        - A player's scoring minutes

        Logistic Regression was selected as the final retrieval model
        because it achieved better manual robustness than the CNN.
        Both model predictions can still be compared.
        """
    )

    gr.Markdown(
        "## Official Model Evaluation"
    )

    gr.Dataframe(
        value=benchmark_results_df,
        headers=[
            "Model",
            "Generated-Test Accuracy",
            "Generated-Test Macro F1",
            "Manual Accuracy",
            "Manual Macro F1"
        ],
        label="Model Performance",
        interactive=False
    )

    gr.Markdown(
        """
        Both models achieved 100% generated-test accuracy.
        Logistic Regression achieved 74% manual accuracy, compared
        with 66% for the CNN. Confidence is not the same as accuracy.
        """
    )

    gr.Markdown(
        "## Ask a Football Question"
    )

    question_input = gr.Textbox(
        lines=3,
        label="Football Question",
        placeholder=(
            "Example: Who won between Brazil and Germany "
            "on 8 July 2014?"
        )
    )

    model_selector = gr.Radio(
        choices=[
            "Logistic Regression",
            "1D CNN",
            "Compare Both"
        ],
        value="Compare Both",
        label="Intent Predictions to Display"
    )

    answer_button = gr.Button(
        "Get Football Answer",
        variant="primary"
    )

    factual_answer_output = gr.Textbox(
        label="Factual Answer",
        interactive=False,
        lines=3
    )

    prediction_output = gr.Dataframe(
        headers=[
            "Model",
            "Predicted Intent",
            "Confidence"
        ],
        datatype=[
            "str",
            "str",
            "number"
        ],
        label="Intent Prediction Results",
        interactive=False
    )

    retrieval_details_output = gr.Markdown(
        label="Retrieval Details"
    )

    answer_button.click(
        fn=run_integrated_football_qa,
        inputs=[
            question_input,
            model_selector
        ],
        outputs=[
            prediction_output,
            factual_answer_output,
            retrieval_details_output
        ]
    )

    question_input.submit(
        fn=run_integrated_football_qa,
        inputs=[
            question_input,
            model_selector
        ],
        outputs=[
            prediction_output,
            factual_answer_output,
            retrieval_details_output
        ]
    )

    gr.Examples(
        examples=[
            [
                (
                    "Who won between Brazil and Germany "
                    "on 8 July 2014?"
                ),
                "Compare Both"
            ],
            [
                (
                    "What was the score when Brazil played "
                    "Germany on 2014-07-08?"
                ),
                "Compare Both"
            ],
            [
                (
                    "Who scored when Brazil faced Germany "
                    "on 8 July 2014?"
                ),
                "Compare Both"
            ],
            [
                (
                    "How many goals did Lionel Messi score "
                    "when Argentina played France on "
                    "18 December 2022?"
                ),
                "Compare Both"
            ],
            [
                (
                    "At what minutes did Kylian Mbappé score "
                    "for France against Argentina on "
                    "18 December 2022?"
                ),
                "Compare Both"
            ],
            [
                (
                    "Who won between Brazil and Germany?"
                ),
                "Compare Both"
            ]
        ],
        inputs=[
            question_input,
            model_selector
        ]
    )

demo.launch(
    share=True,
    debug=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8fe71144bf11d5b0bf.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
